# Course 4 — Cross-Validation

One train/test split is noisy. K-fold cross-validation is how you
stop fooling yourself when tuning a hyperparameter.

**Sections**
1. The validation-set trap (0:00–0:30)
2. LOOCV and K-fold (0:30–1:00)
3. The bootstrap (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold, LeaveOneOut
from sklearn.metrics import mean_squared_error
rng = np.random.default_rng(0)
auto = load_dataset('auto')
x = auto[['horsepower']].to_numpy()
y = auto['mpg'].to_numpy()


## 1. The validation-set trap

### Pick the polynomial degree with one split — repeat ten times

In [ ]:
degrees = range(1, 11)
test_mse_runs = np.empty((10, len(degrees)))
for r, seed in enumerate(range(10)):
    xtr, xte, ytr, yte = train_test_split(x, y, test_size=0.5, random_state=seed)
    for j, d in enumerate(degrees):
        pipe = Pipeline([('p', PolynomialFeatures(d, include_bias=False)),
                         ('lr', LinearRegression())])
        pipe.fit(xtr, ytr)
        test_mse_runs[r, j] = mean_squared_error(yte, pipe.predict(xte))

fig, ax = plt.subplots(figsize=(7, 4))
for r in range(10):
    ax.plot(degrees, test_mse_runs[r], color='gray', linewidth=1, alpha=0.7)
ax.set_xlabel('polynomial degree'); ax.set_ylabel('held-out MSE')
ax.set_title('10 different validation splits → 10 different "best" degrees')
plt.show()


Each gray line is one random split. The minimum jumps around — that is
noise, not signal. We need to average over splits.

## 2. LOOCV and K-fold

### K-fold via `cross_val_score`

In [ ]:
cv_mse = []
for d in degrees:
    pipe = Pipeline([('p', PolynomialFeatures(d, include_bias=False)),
                     ('lr', LinearRegression())])
    scores = cross_val_score(pipe, x, y, cv=KFold(n_splits=10, shuffle=True, random_state=0),
                              scoring='neg_mean_squared_error')
    cv_mse.append(-scores.mean())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(degrees), cv_mse, marker='o')
best = int(np.argmin(cv_mse)) + 1
ax.axvline(best, color='C3', linestyle='--', label=f'best d = {best}')
ax.set_xlabel('degree'); ax.set_ylabel('10-fold CV MSE'); ax.legend()
ax.set_title('ISLP Fig 5.4 style: CV picks a stable winner'); plt.show()


### LOOCV — k equals n

In [ ]:
# Use degree 2 to make it cheap (n = 392 fits).
pipe = Pipeline([('p', PolynomialFeatures(2, include_bias=False)),
                 ('lr', LinearRegression())])
loo_scores = cross_val_score(pipe, x, y, cv=LeaveOneOut(),
                              scoring='neg_mean_squared_error')
print(f'LOOCV MSE (d=2) = {-loo_scores.mean():.4f}')


## 3. The bootstrap

### Estimate SE of a regression coefficient by resampling

In [ ]:
n = len(x)
B = 1000
slopes = np.empty(B)
for k in range(B):
    idx = rng.choice(n, n, replace=True)
    m = LinearRegression().fit(x[idx], y[idx])
    slopes[k] = m.coef_[0]
print(f'bootstrap mean slope = {slopes.mean():.4f}')
print(f'bootstrap SE        = {slopes.std(ddof=1):.4f}')
lo, hi = np.quantile(slopes, [0.025, 0.975])
print(f'95% CI              = [{lo:.4f}, {hi:.4f}]')


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(slopes, bins=40, alpha=0.7); ax.set_xlabel('bootstrapped slope')
ax.set_title('1000 resamples — sampling distribution of the slope'); plt.show()


### Preview: stratified folds for classification

Next week, when `y` is a class label, K-fold can accidentally put all
of one class in one fold. `StratifiedKFold` preserves the class
proportions in every fold. Same `cross_val_score` API.

### Recap
- One split is noise; K-fold averages out the noise.
- LOOCV is just K-fold with K = n. Expensive, low bias, high variance.
- The bootstrap gives a CI for any statistic without distributional
  assumptions — just resample the data with replacement.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** On `iris` *as regression* (predict `petal_length` from
`sepal_length`, `sepal_width`, `petal_width`), compute the 5-fold CV
MSE for plain linear regression.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
iris = load_dataset('iris')
X = iris[['sepal_length', 'sepal_width', 'petal_width']]
y = iris['petal_length']
scores = cross_val_score(LinearRegression(), X, y,
                          cv=KFold(5, shuffle=True, random_state=0),
                          scoring='neg_mean_squared_error')
print(f'5-fold CV MSE = {-scores.mean():.4f} ± {scores.std():.4f}')


---

## Exercise 2

**Task 2.** Bootstrap a 95% CI for the coefficient on `petal_width` in
that model. Use 1000 resamples.

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = X.to_numpy(); yy = y.to_numpy(); n = len(x)
j = list(X.columns).index('petal_width')
coefs = np.empty(1000)
for k in range(1000):
    idx = rng.choice(n, n, replace=True)
    coefs[k] = LinearRegression().fit(x[idx], yy[idx]).coef_[j]
lo, hi = np.quantile(coefs, [0.025, 0.975])
print(f'95% CI on petal_width coef: [{lo:.3f}, {hi:.3f}]')


---

## Exercise 3

**Task 3.** *Gotcha.* Tune polynomial degree with CV on the *whole*
dataset, then report training-set R² for the chosen model. Why is that
training-set number misleading? (Hint: degree was chosen using these
rows.)

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
best_d, best_mse = None, float('inf')
for d in range(1, 6):
    pipe = Pipeline([('p', PolynomialFeatures(d, include_bias=False)),
                     ('lr', LinearRegression())])
    s = -cross_val_score(pipe, X, y, cv=5, scoring='neg_mean_squared_error').mean()
    if s < best_mse:
        best_d, best_mse = d, s
final = Pipeline([('p', PolynomialFeatures(best_d, include_bias=False)),
                  ('lr', LinearRegression())]).fit(X, y)
print(f'CV picked d = {best_d}; training R^2 = {final.score(X, y):.4f}')
print('This R^2 is on data the model already saw — inflated. For an honest')
print('estimate, use NESTED CV: outer fold for evaluation, inner fold for tuning.')
